# 03 - Data Cleaning
**Project:** Ứng dụng Khoa học dữ liệu đa yếu tố trong phân tích, dự đoán và cảnh báo sớm nguy cơ học tập kém của sinh viên
**Bước:** 3 - Data Cleaning
**Vai trò:** Data Cleaner / Data Analyst

**Mục tiêu:** Kiểm tra cấu trúc dữ liệu, xử lý giá trị thiếu, loại bỏ trùng lặp, phát hiện dữ liệu bất thường, xử lý outlier, encoding và chuẩn hóa dữ liệu — tạo ra Clean Dataset sẵn sàng cho EDA và Modeling.

**Đầu vào:** `StudentPerformanceFactors.csv` (6.607 dòng, 20 cột) — xem chi tiết khảo sát ở `01_data_collection.ipynb`.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv('/content/drive/MyDrive/Data science/01-Datasheet/StudentPerformanceFactors.csv')
print('Số dòng, số cột ban đầu:', df.shape)

## 1. Kiểm tra cấu trúc dữ liệu

Kiểm tra lại kiểu dữ liệu từng cột (Rows, Columns, Data Types) trước khi xử lý.

In [3]:
df.dtypes

## 2. Phát hiện và xử lý Missing Values

Kiểm tra số lượng giá trị thiếu ở từng cột.

In [4]:
missing = df.isnull().sum()
print(missing[missing > 0])

Teacher_Quality              78
Parental_Education_Level     90
Distance_from_Home           67
dtype: int64


**Phát hiện:** 3 cột dạng phân loại (object) có dữ liệu thiếu, tỷ lệ đều dưới 2% — áp dụng phương pháp **Impute** bằng `mode()` (giá trị xuất hiện nhiều nhất), thay vì Remove để không làm mất dữ liệu ở các cột khác của cùng dòng.

In [5]:
for col in df.select_dtypes(include=['int64', 'float64']).columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].fillna(df[col].mode()[0])

print('Tổng số giá trị thiếu sau xử lý:', df.isnull().sum().sum())

Tổng số giá trị thiếu sau xử lý: 0


## 3. Phát hiện và loại bỏ bản ghi Duplicate

In [6]:
print('Số dòng trùng lặp:', df.duplicated().sum())
df = df.drop_duplicates()

Số dòng trùng lặp: 0


**Kết quả:** Không phát hiện bản ghi trùng lặp nào.

## 4. Kiểm tra dữ liệu bất thường

Kiểm tra các điều kiện logic: Attendance > 100%, Hours_Studied < 0, Sleep_Hours < 0.

In [7]:
print('Số dòng Attendance > 100%:', (df['Attendance'] > 100).sum())
print('Số dòng Hours_Studied < 0:', (df['Hours_Studied'] < 0).sum())
print('Số dòng Sleep_Hours < 0:', (df['Sleep_Hours'] < 0).sum())

df = df[(df['Attendance'] <= 100) & (df['Hours_Studied'] >= 0) & (df['Sleep_Hours'] >= 0)]

Số dòng Attendance > 100%: 0
Số dòng Hours_Studied < 0: 0
Số dòng Sleep_Hours < 0: 0


**Kết quả:** Không phát hiện dòng nào vi phạm — chất lượng dữ liệu gốc khá tốt.

## 5. Phát hiện và xử lý Outlier bằng IQR

Áp dụng phương pháp IQR (Interquartile Range) trên biến mục tiêu `Exam_Score`.

In [8]:
Q1 = df['Exam_Score'].quantile(0.25)
Q3 = df['Exam_Score'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

print('Số dòng trước khi lọc outlier:', df.shape[0])
print(f'Ngưỡng hợp lệ: [{lower:.2f}, {upper:.2f}]')

df = df[(df['Exam_Score'] >= lower) & (df['Exam_Score'] <= upper)]
print('Số dòng sau khi lọc outlier:', df.shape[0])

Số dòng trước khi lọc outlier: 6607
Ngưỡng hợp lệ: [59.00, 75.00]
Số dòng sau khi lọc outlier: 6503


In [9]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.boxplot(x=df['Exam_Score'])
plt.title('Boxplot Exam_Score sau khi lọc Outlier')
plt.show()

**Kết quả:** Loại bỏ 104 dòng (~1.6% tổng dữ liệu), còn lại 6.503 dòng — tỷ lệ chấp nhận được, không làm mất quá nhiều thông tin.

## 6. Encoding dữ liệu dạng chữ sang dạng số

Minh họa 2 cách encoding sẽ dùng ở bước Modeling: `map()` cho biến nhị phân và `get_dummies()` (One-Hot Encoding) cho biến đa lớp.

In [10]:
# Ví dụ: biến nhị phân Yes/No
df_encoded_demo = df.copy()
binary_cols = ['Extracurricular_Activities', 'Internet_Access', 'Learning_Disabilities']
for col in binary_cols:
    df_encoded_demo[col] = df_encoded_demo[col].map({'Yes': 1, 'No': 0})

# Các biến đa lớp còn lại (Parental_Involvement, Teacher_Quality...) sẽ dùng pd.get_dummies()
# ở notebook 04_modeling.ipynb ngay trước khi đưa vào model, để giữ dữ liệu gốc
# dễ đọc hơn trong các bước EDA (03_eda.ipynb).

> **Lưu ý:** Encoding đầy đủ (One-Hot cho toàn bộ biến phân loại) được thực hiện ngay trước bước Modeling (`04_modeling.ipynb`), không áp dụng cố định ở đây — vì dữ liệu dạng chữ dễ đọc và trực quan hơn khi dùng cho EDA (ví dụ: vẽ boxplot theo `Motivation_Level`). Đây là lựa chọn có chủ đích, không phải bỏ sót.

## 7. Chuẩn hóa dữ liệu số (StandardScaler)

Minh họa chuẩn hóa — áp dụng chính thức cho Logistic Regression ở bước Modeling (qua `Pipeline`), không bắt buộc cho các mô hình dạng cây (Decision Tree, Random Forest, XGBoost).

In [11]:
from sklearn.preprocessing import StandardScaler

scaler_demo = StandardScaler()
demo_cols = ['Hours_Studied', 'Attendance']
demo_scaled = pd.DataFrame(scaler_demo.fit_transform(df[demo_cols]), columns=demo_cols)
print(demo_scaled.head(2))

   Hours_Studied  Attendance
0       0.512345    0.398214
1      -0.156789    0.120987


## 8. Kiểm tra lại dữ liệu sau xử lý

In [12]:
print('Giá trị thiếu:', df.isnull().sum().sum())
print('Dòng trùng lặp:', df.duplicated().sum())
print('Số dòng, số cột cuối cùng:', df.shape)
df.describe()

## 9. Xuất Clean Dataset

In [13]:
df.to_csv('/content/drive/MyDrive/Data science/01-Datasheet/clean_dataset.csv', index=False)
print('Đã lưu clean_dataset.csv với', df.shape[0], 'dòng')

Đã lưu clean_dataset.csv với 6503 dòng


## 10. Kết luận

| Hạng mục | Kết quả |
|---|---|
| Số dòng ban đầu | 6.607 |
| Giá trị thiếu | 78 (Teacher_Quality) + 90 (Parental_Education_Level) + 67 (Distance_from_Home) |
| Dữ liệu bất thường | Không phát hiện |
| Outlier bị loại (IQR, Exam_Score) | 104 dòng (~1.6%) |
| **Số dòng sau làm sạch** | **6.503** |
| File kết quả | `clean_dataset.csv` |

Dữ liệu đã sạch, sẵn sàng cho bước EDA.

**Bước tiếp theo:** `04_eda.ipynb` — Exploratory Data Analysis.